# Lab 6.13 (Bonus) &mdash; AutoGPT-style autonomy vs LangGraph control

**Level:** Bonus / Advanced &nbsp;|&nbsp; **Est. time:** ~30 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

> This bonus is the **hands-on for the two slides** *AutoGPT architecture* and *AutoGPT vs LangGraph*. It is **not** one of the 12 graded labs &mdash; it is a **runnable demo** (already filled in): run it top to bottom, read the two traces, then do the **Your turn** edits so the lesson is felt, not just told.

### What you'll do
- Run a **minimal AutoGPT-style self-prompting loop** &mdash; one LLM, a fixed THOUGHT/COMMAND schema, tools, and a while-loop.
- Run the **same goal** through a **bounded LangGraph agent** (`create_agent` + `recursion_limit`).
- **Read the two real traces** and see the difference &mdash; not *which is smarter* (same goal, same tools) but **who authors the flow and what enforces the bounds**.

## Concept &mdash; two ways to run the loop

**AutoGPT** is *one LLM wrapped in a self-prompting loop*: it forces the model to emit a `THOUGHT` and a single `COMMAND` each turn, runs that command as a tool, feeds the result back, and repeats &mdash; **by default with no human and no fixed stop**. The structure lives in a **prompt + a while-loop**.

**LangGraph / `create_agent`** is the opposite: **you** author the flow and the bounds (`recursion_limit`, which tools are bound, where a human approves).

We give **both** the same tools and the same goal &mdash; *add up four quarterly figures, then save a one-line report* &mdash; so what differs is **structure**.

> **Heads-up on models &mdash; this pairing *is* part of the lesson.** The AutoGPT **text loop** needs a model that follows a plain-text protocol: we use the local **`llama3.1:8b`**. The LangGraph agent needs a **tool-calling** model: we use Groq **`gpt-oss-20b`**. Swap them and each fumbles the other's job (gpt-oss-20b *refuses* the raw text protocol; llama3.1:8b fumbles structured tool-calls on a compound goal). **Tool-calling models + graphs are exactly what replaced text-loop AutoGPT** &mdash; the paradigm and the model co-evolved.

In [ ]:
# Setup -- run me first
import os, socket, ast, operator, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"))   # GROQ_API_KEY for Part B (Groq is the Day 4-5 provider)

WORK = "/tmp/biaa-lab-06-13"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If down, start:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

def groq_ready():
    """True if a Groq API key is available (free at console.groq.com; put it in .env)."""
    return bool(os.getenv("GROQ_API_KEY"))

from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
# Each paradigm on the model it was built for (see the note above):
#   Part A (AutoGPT text loop)  -> local text model. Pin 127.0.0.1 -- "localhost" can misroute.
#   Part B (LangGraph agent)    -> Groq tool-calling model.
llm_local = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")
llm_groq  = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# One AST-safe calculator, shared by both approaches (never bare eval).
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def _ev(node):
    if isinstance(node, ast.Constant): return node.value
    if isinstance(node, ast.BinOp):    return _OPS[type(node.op)](_ev(node.left), _ev(node.right))
    if isinstance(node, ast.UnaryOp):  return _OPS[type(node.op)](_ev(node.operand))
    raise ValueError("unsupported expression")
def calc(expr):
    return _ev(ast.parse(expr, mode="eval").body)

GOAL = ("Add up the 2023 quarterly revenue Q1=120, Q2=135, Q3=150, Q4=160 (in $M), "
        "then save a one-line report of the total.")

print("Ollama up:", ollama_up(), "(Part A) | Groq key:", groq_ready(), "(Part B) | WORK:", WORK)
if not ollama_up(): print("  Part A needs Ollama:  ollama run llama3.1:8b")
if not groq_ready(): print("  Part B needs GROQ_API_KEY in .env")

## Part A &mdash; the AutoGPT pattern, from scratch

A self-prompting loop: each turn we ask the model for a `THOUGHT` and one `COMMAND`, run the command, and feed the observation back. Real AutoGPT has **no step cap** &mdash; we add a small `MAX_STEPS` only so the notebook ends. Driven by the local **`llama3.1:8b`**, which follows the text protocol well.

Read the two helpers &mdash; `parse_autogpt` pulls the `COMMAND` line out of the reply, `run_command` dispatches it to a tool &mdash; then run the loop.

In [ ]:
AUTOGPT_SYS = (
    "You are an autonomous agent. Available commands:\n"
    "  calculator: <arithmetic expression>\n"
    "  save_report: <one line of text>\n"
    "  finish: <final answer>\n"
    "Each turn reply EXACTLY as:\nTHOUGHT: <reasoning>\nCOMMAND: <one command from the list>"
)

def parse_autogpt(raw):
    """Pull the THOUGHT and the single COMMAND line out of the model's reply."""
    thought, command = "", ""
    for line in raw.splitlines():
        u = line.strip().upper()
        if u.startswith("THOUGHT:"):
            thought = line.split(":", 1)[1].strip()
        elif u.startswith("COMMAND:"):
            command = line.split(":", 1)[1].strip()
    return thought, command

def run_command(command):
    """Dispatch one COMMAND to a tool. A tool must NEVER raise -- that would abort the loop."""
    try:
        name, _, arg = command.partition(":")
        name, arg = name.strip().lower(), arg.strip()
        if name == "calculator":
            return f"{arg} = {calc(arg)}"
        if name == "save_report":
            open(os.path.join(WORK, "report.txt"), "w").write(arg)
            return f"saved to {os.path.join(WORK, 'report.txt')}"
        return f"unknown command: {name}"
    except Exception as e:
        return f"error: {e}"

print("helpers ready")

In [ ]:
# Run the AutoGPT-style loop for real (local llama3.1:8b).
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
  try:
    MAX_STEPS = 8          # AutoGPT has NO cap by default; we add one so the notebook ends
    history = f"GOAL: {GOAL}\n"
    finished = False
    for step in range(1, MAX_STEPS + 1):
        raw = llm_local.invoke(AUTOGPT_SYS + "\n\n" + history + "\nYour move:").content
        thought, command = parse_autogpt(raw)
        print(f"[step {step}] THOUGHT: {thought[:80]}")
        print(f"[step {step}] COMMAND: {command[:80]}")
        if command.strip().lower().startswith("finish"):
            print("  -> model called finish. DONE.\n"); finished = True; break
        obs = run_command(command)
        print(f"  OBS: {obs[:120]}")
        history += f"\nSTEP {step}: {command}\nOBSERVATION: {obs}\n"
    if not finished:
        print("  -> hit MAX_STEPS without a clean finish -- the classic unbounded-loop failure.")
  except Exception as e:
    print("Run error:", type(e).__name__, e)

**Read your trace.** It probably reached the answer &mdash; the model is capable. But notice what nothing enforced: **you** had to bolt on `MAX_STEPS` (nothing else stops it), and the model could name **any** command &mdash; the loop just dispatches whatever text it emits. Run the next cell to see that plainly.

In [ ]:
# The loop dispatches whatever command the model NAMES. It is safe here only because we never
# implemented a dangerous command -- nothing in the loop *restricts* what may run.
print(run_command("delete: /tmp/important-file"))   # -> "unknown command: delete"
print("Safety here = your dispatch code. A real AutoGPT with a shell/file tool would run it.")

## Part B &mdash; the same goal, bounded with LangGraph

Now hand the **same goal and tools** to `create_agent` (a LangGraph agent), driven by Groq **`gpt-oss-20b`** (a tool-calling model). You **bind** exactly the tools it may use and cap the loop with `recursion_limit` &mdash; so misbehaviour isn't *trusted away*, it's made **impossible**. Run it and read the real message trace.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def calculator(expression: str) -> str:
    """Compute a basic arithmetic expression like '120+135' and return the result."""
    try:
        return f"{expression} = {calc(expression)}"
    except Exception as e:
        return f"error: {e}"

@tool
def save_report(text: str) -> str:
    """Save a one-line report string to the working directory; returns the file path."""
    try:
        open(os.path.join(WORK, "report.txt"), "w").write(text)
        return f"saved to {os.path.join(WORK, 'report.txt')}"
    except Exception as e:
        return f"error: {e}"

def print_trace(result):
    """Print the REAL agent message trace: tool calls, observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:160])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:240])

print("tools defined: calculator, save_report")

In [ ]:
# Build the bounded agent and run it for real (Groq gpt-oss-20b).
if not groq_ready():
    print("No GROQ_API_KEY -- set it in .env, then re-run this cell.")
else:
    try:
        agent = create_agent(llm_groq, tools=[calculator, save_report])   # bind ONLY these tools (least privilege)
        cfg = {"recursion_limit": 8}                                      # cap the loop
        result = agent.invoke({"messages": [("user", GOAL)]}, cfg)
        print_trace(result)
    except Exception as e:
        print("Run error:", type(e).__name__, e)

## Compare

Both reach **$565M** on the same goal &amp; tools. The difference isn't IQ, it's **control**:

| | AutoGPT-style loop (Part A) | LangGraph agent (Part B) |
|---|---|---|
| Who authored the flow | the **model**, at run time | **you**, up front |
| What bounds the loop | only the `MAX_STEPS` you bolted on | `recursion_limit`, a first-class cap |
| Tools it *may* call | any it names in text (you hope your dispatch is safe) | only the tools you **bound** &mdash; others are impossible |
| When it stops | when the model *says* `finish` | when the graph reaches `END` |
| Failure at scale | drift, repeats, runaway cost | predictable &mdash; you see &amp; steer each node |

On a toy task the unbounded loop is fine. At real scale, *trusting the model to behave* is where drift, cost and runaway loops live (Module 10) &mdash; which is why your **capstone is a bounded, guardrailed graph**, not an unleashed AutoGPT. *You earn autonomy with structure.*

### Your turn
- Raise `MAX_STEPS` to 20 in Part A, and delete the `finish` line from `AUTOGPT_SYS` &mdash; now nothing tells it to stop. Watch it burn every step.
- Add a `delete` branch to `run_command` and put `delete: ...` in reach &mdash; the loop will run it. The LangGraph agent **can't**: there's no such tool bound (least privilege).
- Add a human-approval step to Part B's flow (see Lab 6.8, the LangGraph state machine).